In [188]:
import pandas as pd
from datetime import datetime, date
from calendar import monthrange


In [189]:
today = date.today()

first_day = today.replace(day=1)

last_day = today.replace(
    day=monthrange(today.year, today.month)[1]
)

# last_day = '2026-08-16'

In [190]:
sub_types = [
    'beIN Quartar Installment',
    'CNE Subscriber',
    'MCE staff (CNE staff)',
    'BeIN sports CC',
    'beIN Bi Installment',
    'Corporate Subscriber', 
    'Bein NC',
    'Bulk DTH customer'
    ]

In [191]:
bein_filename = input(f'Enter beindata for first day in month' )
bein_src = pd.read_csv(bein_filename, dtype='str')
ft_src = pd.read_csv(r"S:\18.08.26\30761933_NEWCNEFINTRANSRPT.CSV", dtype='str', on_bad_lines='skip')

In [192]:
bein = bein_src.copy()
print('Base:', bein.shape)


# bein.to_csv('bein active check.csv', index= False)
bein = bein.loc[bein['Status']=='Active']
print('after filter Active:', bein.shape)

bein = bein.loc[bein['Customer Type'].isin(sub_types)]
print('after filter types:', bein.shape)

bein = bein.sort_values('End Date', ascending=False)
bein = bein.drop_duplicates(subset=['Customer Number'], keep='first')
print('after remove duplicates:', bein.shape)

bein['End Date'] = pd.to_datetime(bein['End Date'],format='%d-%m-%Y', dayfirst= True)
bein.to_csv('bein test 8.csv', index=False)
bein = bein.loc[bein['End Date'].between(pd.to_datetime(first_day),pd.to_datetime(last_day))]

print('after filter dates:', bein.shape)
ft = ft_src.copy()
ft= ft.loc[(ft['Doc Status']=='Posted') & (ft['Doc Type'].isin(['Invoice']))]
ft['Created Date'] = pd.to_datetime(ft['Created Date'], dayfirst=True)
ft = ft.loc[ft['Created Date'].between(pd.to_datetime(first_day),pd.to_datetime(last_day))]

bein['renewed'] = 'Not Yet'

bein.loc[bein['Customer Number'].isin(ft['Subscriber Nr']),'renewed'] = 'Yes'

bein = pd.merge(left=bein, right=ft[['Subscriber Nr','Created Date']], left_on='Customer Number', right_on='Subscriber Nr', how='left')

week_num_disconnection = (((bein['End Date'].dt.day - 1) // 7) + 1).clip(upper=4)
week_num_renewed = (((bein['Created Date'].dt.day - 1) // 7) + 1).clip(upper=4)

bein['disconnection week'] = 'W' + week_num_disconnection.astype(str).replace(".0","")
bein['renewed week'] = 'W' + week_num_renewed.astype(str).replace(".0","")
bein['renewed week'] = bein['renewed week'].replace('Wnan','')

bein['renewed'].value_counts()

# renewed = ft.loc[ft['Subscriber Nr'].isin(bein['Customer Number'])].drop_duplicates(subset=['Subscriber Nr'])
# renewed.shape

# bein.to_csv('monthly disconnection.csv', index=False)

Base: (6080334, 17)
after filter Active: (355170, 17)
after filter types: (339530, 17)
after remove duplicates: (168615, 17)
after filter dates: (20563, 17)


C:\Users\mturky\AppData\Local\Temp\1\ipykernel_33052\956579009.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ft['Created Date'] = pd.to_datetime(ft['Created Date'], dayfirst=True)


renewed
Not Yet    15480
Yes         5150
Name: count, dtype: int64